# EYES-DEFY-ANEMIA — Segmentation Phase 2 sweep: STRONG tier (6 of 18 combos, heaviest)

One of **3 tier-specific notebooks** (Base / Mid / Strong) — see `segmentation-pretrained-sweep-base.ipynb` in this same folder for the full rationale (a real Kaggle disk-quota crash on the original combined 18-combo notebook, plus the fixes applied since: single fp16 checkpoint per model instead of up to 3 fp32 ones, and a `sync_outputs()` that no longer leaves duplicate copies behind).

**This notebook trains the 3 heaviest architectures** (TransUNet ~120.9M, ConvNeXt-Large U-Net ~203.3M, Swin-Large + UperNet ~233.9M), each on both tissue types — 6 combos total, combo numbers 13-18 of the full 18. These are also the models most likely to test GPU memory limits (see the GPU-OOM note below) — running them in their own session, after Base/Mid have already finished, keeps this the only notebook where that risk applies. Full roster/rationale: `Segmentation/.project_memory/04_pretrained_architecture_sweep.md`.

## Setup

In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


In [2]:
# rm -rf first so a re-run within the same kernel session stays idempotent
# instead of nesting a second clone inside the first.
!rm -rf eyes-defy-anemia
!git clone https://github.com/manivafapour/eyes-defy-anemia.git
%cd eyes-defy-anemia

Cloning into 'eyes-defy-anemia'...
remote: Enumerating objects: 1142, done.
remote: Counting objects: 100% (823/823), done.
remote: Compressing objects: 100% (521/521), done.
remote: Total 1142 (delta 410), reused 671 (delta 294), pack-reused 319 (from 1)
Receiving objects: 100% (1142/1142), 76.62 MiB | 7.00 MiB/s, done.
Resolving deltas: 100% (573/573), done.
/kaggle/working/eyes-defy-anemia


In [3]:
# Diagnostic, not a hardcoded assumption -- Kaggle's actual dataset mount
# path does not always match its display name, and can be nested deeper
# than expected (this project has been bitten by this before, twice now:
# see classification/.project_memory/kaggle/01_kaggle_notes.md -- real
# paths have landed under /kaggle/input/datasets/<username>/<slug>/, not
# directly under /kaggle/input/<slug>/). Recurses a few levels deep so
# this is caught in one pass instead of needing to descend manually.
# Read the output below, THEN set the dataset dir variable(s) in the next cell.
import os


def print_tree(path, depth=0, max_depth=4):
    for entry in sorted(os.listdir(path)):
        full = os.path.join(path, entry)
        print("  " * depth + entry)
        if os.path.isdir(full) and depth < max_depth:
            print_tree(full, depth + 1, max_depth)


print_tree("/kaggle/input")

datasets
  manivafapour21
    aligned-raw
      aligned_raw
        alignment_log.csv
        images
        masks
    aligned-raw-forniceal
      aligned_raw_forniceal
        alignment_log.csv
        images
        masks


**Before running the next cell:** attach your uploaded dataset(s) to this notebook -- either both zips together as ONE Kaggle dataset, or as TWO SEPARATE datasets (this is what actually happened the first time this notebook was run: Kaggle listed them individually in the Input panel as `aligned_raw` and `aligned_raw_forniceal`). Either way, run the listing cell above, read its REAL printed output, and set the path(s) below from that -- **not** the placeholder text left in by default, and not a guessed path based on the dataset's display name (Kaggle's actual mount path does not always match it).

In [4]:
# Confirmed real mount paths (project author's Kaggle account, verified via
# the print_tree() listing above on 2026-08-08 -- see
# Segmentation/.project_memory/kaggle/01_kaggle_notes.md for the full
# doubly-nested structure this came from). If you re-attach the datasets
# under a different username/slug, or Kaggle changes its mount scheme again,
# re-run the listing cell above and update these two lines from its real
# output -- don't guess.
ALIGNED_RAW_DATASET_DIR = "/kaggle/input/datasets/manivafapour21/aligned-raw"
ALIGNED_RAW_FORNICEAL_DATASET_DIR = "/kaggle/input/datasets/manivafapour21/aligned-raw-forniceal"

In [5]:
# Only packages actually missing from Kaggle's base image (torch/torchvision,
# opencv, pandas, PIL, scikit-learn -- and therefore scipy, its own
# dependency -- are already there; scipy is listed explicitly anyway
# below rather than silently assumed, since it is now a real, load-bearing
# dependency for HD95/Wilcoxon in segmentation_metrics.py /
# compare_models_significance.py). Deliberately NOT
# `pip install -r requirements.txt` -- that file is pinned to the local
# Windows/CUDA 13.0 build and would try to reinstall Kaggle's own correctly
# configured GPU PyTorch with an incompatible build.
!pip install -q optuna albumentations scipy segmentation-models-pytorch timm transformers einops

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 10.2 MB/s eta 0:00:00


## Data

In [6]:
import shutil
import zipfile
from pathlib import Path

DST_ROOT = Path("Segmentation/data/processed")


def stage_tissue_data(name: str, dataset_dir: str):
    """Copies {name}/ from the given Kaggle-attached dataset directory into
    Segmentation/data/processed/{name}/. Tries three possible layouts rather
    than assuming one, since Kaggle can present an uploaded zip differently
    depending on upload method:
      1. dataset_dir/{name}/images,masks/  -- the zip's own internal "{name}/"
         prefix preserved as-is (this is how aligned_raw.zip/
         aligned_raw_forniceal.zip were actually built -- see
         Segmentation/scripts/build_aligned_dataset{,_forniceal}.py).
      2. dataset_dir/images,masks/         -- Kaggle stripped/flattened that
         top-level folder on extraction.
      3. dataset_dir/{name}.zip            -- never auto-extracted at all,
         still sitting there as a raw zip file.
    """
    dataset_dir = Path(dataset_dir)
    dst = DST_ROOT / name
    shutil.rmtree(dst, ignore_errors=True)

    nested_dir = dataset_dir / name
    flat_zip = dataset_dir / f"{name}.zip"

    if nested_dir.is_dir():
        shutil.copytree(nested_dir, dst)
    elif (dataset_dir / "images").is_dir() and (dataset_dir / "masks").is_dir():
        shutil.copytree(dataset_dir, dst)
    elif flat_zip.is_file():
        dst.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(flat_zip) as zf:
            zf.extractall(DST_ROOT)  # zip's own internal paths already start with f"{name}/"
    else:
        raise FileNotFoundError(
            f"Could not find {name}/, images+masks/, or {name}.zip under {dataset_dir} -- "
            f"run the /kaggle/input listing cell above and check what's actually there."
        )

    n_images = len(list((dst / "images").glob("*.jpg")))
    n_masks = len(list((dst / "masks").glob("*.png")))
    print(f"{name}: {n_images} images, {n_masks} masks staged at {dst}")


stage_tissue_data("aligned_raw", ALIGNED_RAW_DATASET_DIR)
stage_tissue_data("aligned_raw_forniceal", ALIGNED_RAW_FORNICEAL_DATASET_DIR)

aligned_raw: 201 images, 201 masks staged at Segmentation/data/processed/aligned_raw
aligned_raw_forniceal: 211 images, 211 masks staged at Segmentation/data/processed/aligned_raw_forniceal


In [7]:
# Combined sanity check, run BEFORE any real training:
#   1. Confirm the pip-installed heavy dependencies actually work -- a
#      dataloader-only check would NOT catch a missing/broken install here
#      (classification's own Kaggle notes: a plain dataloader check only
#      exercises dataset.py's imports, so a missing package silently
#      surfaces only much later, on the first real training script).
#   2. Confirm the 9-model registry itself imports cleanly.
#   3. Pull one real batch from BOTH tissue-type dataloaders.
import sys
from pathlib import Path

sys.path.insert(0, str(Path("Segmentation/scripts").resolve()))
sys.path.insert(0, str(Path("Segmentation").resolve()))

import segmentation_models_pytorch as smp
import timm
import transformers
import einops

print("segmentation_models_pytorch", smp.__version__)
print("timm", timm.__version__)
print("transformers", transformers.__version__)
print("einops", einops.__version__)

from models.segmentation.pretrained_registry import ARCHITECTURE_REGISTRY
print(f"\n{len(ARCHITECTURE_REGISTRY)} architectures registered:")
for name in ARCHITECTURE_REGISTRY:
    print(" ", name)

from dataset import get_dataloaders

loaders = get_dataloaders()
for key in ["aligned_seg_train", "aligned_seg_forniceal_train"]:
    images, masks = next(iter(loaders[key]))
    print(f"\n{key}: image batch {tuple(images.shape)}, mask batch {tuple(masks.shape)}, "
          f"{len(loaders[key].dataset)} patients")

segmentation_models_pytorch 0.5.0
timm 1.0.26
transformers 5.0.0
einops 0.8.2

9 architectures registered:
  cnn_base_efficientnet_b1_unet
  cnn_mid_resnet101_deeplabv3plus
  cnn_strong_convnext_large_unet
  hybrid_base_coatnet0_unet
  hybrid_mid_coatnet2_unet
  hybrid_strong_transunet
  transformer_base_segformer_b2
  transformer_mid_swin_base_upernet
  transformer_strong_swin_large_upernet

aligned_seg_train: image batch (16, 3, 256, 256), mask batch (16, 1, 256, 256), 143 patients

aligned_seg_forniceal_train: image batch (16, 3, 256, 256), mask batch (16, 1, 256, 256), 147 patients


## Training

In [8]:
import shutil
from pathlib import Path


def sync_outputs():
    """Consolidates Segmentation/outputs/{checkpoints,logs,plots}/ into a single
    top-level /kaggle/working/outputs/ folder and re-zips it to
    /kaggle/working/segmentation_sweep_results.zip. Called after EVERY
    training cell below, not just at the end -- if the run gets cut short
    partway through the 18 combos, whatever completed so far is still
    cleanly consolidated and zipped, ready to download.

    IMPORTANT (added after a real Kaggle disk-quota crash, "Your notebook
    tried to use more disk space than is available", 20.93GB used, failed
    5h40m into a real run): each of the 18 training scripts can write up to
    3 full checkpoint files at fp32 (best-overall + one per loss function),
    and none of that is ever deleted between combos -- for the larger
    architectures (ConvNeXt-Large ~203M params, Swin-Large ~234M, etc.)
    those add up to hundreds of MB to ~1GB EACH. Originally this function
    copied Segmentation/outputs/ into /kaggle/working/outputs/ and left the
    source in place, so between the untouched source, the mirrored copy,
    and the zip made from that copy, roughly 3 copies of everything
    produced so far existed on disk simultaneously -- comfortably enough to
    blow a ~20GB quota partway through the Mid/Strong tiers. Now the
    source is deleted right after it's safely copied into
    /kaggle/working/outputs/, which becomes the single accumulating copy
    (plus the zip made from it) -- every training script recreates
    Segmentation/outputs/{checkpoints,logs,plots}/ fresh via its own
    mkdir(parents=True, exist_ok=True) on its next run, so nothing is lost
    by clearing the source here.
    """
    results_dir = Path("/kaggle/working/outputs")
    results_dir.mkdir(parents=True, exist_ok=True)
    for sub in ["checkpoints", "logs", "plots"]:
        src = Path("Segmentation/outputs") / sub
        if src.exists():
            shutil.copytree(src, results_dir / sub, dirs_exist_ok=True)
            shutil.rmtree(src)
    archive_path = shutil.make_archive("/kaggle/working/segmentation_sweep_results", "zip", root_dir=str(results_dir))
    n_files = sum(1 for f in results_dir.rglob("*") if f.is_file())
    print(f"[sync_outputs] {n_files} files consolidated under {results_dir}, zipped to {archive_path}")


sync_outputs()  # harmless no-op now (nothing produced yet), confirms the function works before training starts

[sync_outputs] 0 files consolidated under /kaggle/working/outputs, zipped to /kaggle/working/segmentation_sweep_results.zip


### Training — Strong tier (6 combos, heaviest)

**If a cell below runs out of GPU memory** (Kaggle's T4 has 16GB, vs. the 6GB local GPU this was structurally verified on, where sequentially testing multiple heavy models in one process did hit a CUDA OOM — a local-testing artifact, not necessarily a Kaggle issue, but worth watching for on Swin-Large at 512×512): add a cell before re-running that combo with:
```python
import sys; sys.path.insert(0, "Segmentation/scripts")
import trainer_engine; trainer_engine.BATCH_SIZE = 8
```
This only affects combos run afterward in the same session — anything already completed is unaffected.

In [9]:
# Strong tier, combo 13/18 -- TransUNet (Hybrid, ~120.9M), palpebral
!python Segmentation/scripts/train_pretrained/train_hybrid_strong_transunet_palpebral.py
sync_outputs()

Using device: cuda
Model: hybrid_strong_transunet_palpebral (pretrained build_model)
Dataset: AlignedConjunctivaSegmentationDataset (tissue_type=palpebral)
Image size: 256
[I 2026-08-10 15:39:20,472] A new study created in memory with name: no-name-c7c057ad-1e63-415b-b482-3e2be84e8a6f
model.safetensors: 100%|█████████████████████| 102M/102M [00:02<00:00, 45.7MB/s]
model.safetensors: 100%|█████████████████████| 346M/346M [00:03<00:00, 90.8MB/s]
[hybrid_strong_transunet_palpebral | Trial 0] New best overall val_dice=0.0000 -> saved /kaggle/working/eyes-defy-anemia/Segmentation/outputs/checkpoints/best_hybrid_strong_transunet_palpebral.pth (fp16)
[hybrid_strong_transunet_palpebral | Trial 0 | loss_fn=bce_dice] Epoch  1/30 - train_loss=0.7497 val_loss=0.7639 val_dice=0.0000 val_iou=0.0000 val_precision=1.0000 val_recall=0.0000
[hybrid_strong_transunet_palpebral | Trial 0] New best overall val_dice=0.7913 -> saved /kaggle/working/eyes-defy-anemia/Segmentation/outputs/checkpoints/best_hybrid

In [10]:
# Strong tier, combo 14/18 -- TransUNet (Hybrid, ~120.9M), forniceal_palpebral
!python Segmentation/scripts/train_pretrained/train_hybrid_strong_transunet_forniceal_palpebral.py
sync_outputs()

Using device: cuda
Model: hybrid_strong_transunet_forniceal_palpebral (pretrained build_model)
Dataset: AlignedConjunctivaSegmentationDataset (tissue_type=forniceal_palpebral)
Image size: 256
[I 2026-08-10 16:15:03,734] A new study created in memory with name: no-name-82a3b468-089e-4758-ae6a-c31121cc3368
[hybrid_strong_transunet_forniceal_palpebral | Trial 0] New best overall val_dice=0.7588 -> saved /kaggle/working/eyes-defy-anemia/Segmentation/outputs/checkpoints/best_hybrid_strong_transunet_forniceal_palpebral.pth (fp16)
[hybrid_strong_transunet_forniceal_palpebral | Trial 0 | loss_fn=bce_dice] Epoch  1/30 - train_loss=0.6765 val_loss=0.7407 val_dice=0.7588 val_iou=0.6179 val_precision=0.6745 val_recall=0.8940
[hybrid_strong_transunet_forniceal_palpebral | Trial 0 | loss_fn=bce_dice] Epoch  2/30 - train_loss=0.6161 val_loss=0.6842 val_dice=0.5600 val_iou=0.3986 val_precision=0.3987 val_recall=0.9997
[hybrid_strong_transunet_forniceal_palpebral | Trial 0] New best overall val_dice=0.

In [11]:
# Strong tier, combo 15/18 -- ConvNeXt-Large U-Net (CNN, ~203.3M), palpebral
!python Segmentation/scripts/train_pretrained/train_cnn_strong_convnext_large_unet_palpebral.py
sync_outputs()

Using device: cuda
Model: cnn_strong_convnext_large_unet_palpebral (pretrained build_model)
Dataset: AlignedConjunctivaSegmentationDataset (tissue_type=palpebral)
Image size: 256
[I 2026-08-10 16:51:15,361] A new study created in memory with name: no-name-0da1e514-5194-4ce8-8926-959a9abbcf04
model.safetensors: 100%|██████████████████████| 791M/791M [00:07<00:00, 107MB/s]
[cnn_strong_convnext_large_unet_palpebral | Trial 0] New best overall val_dice=0.1384 -> saved /kaggle/working/eyes-defy-anemia/Segmentation/outputs/checkpoints/best_cnn_strong_convnext_large_unet_palpebral.pth (fp16)
[cnn_strong_convnext_large_unet_palpebral | Trial 0 | loss_fn=bce_dice] Epoch  1/30 - train_loss=0.7446 val_loss=0.8013 val_dice=0.1384 val_iou=0.0747 val_precision=0.0749 val_recall=0.9680
[cnn_strong_convnext_large_unet_palpebral | Trial 0] New best overall val_dice=0.3230 -> saved /kaggle/working/eyes-defy-anemia/Segmentation/outputs/checkpoints/best_cnn_strong_convnext_large_unet_palpebral.pth (fp16)


In [12]:
# Strong tier, combo 16/18 -- ConvNeXt-Large U-Net (CNN, ~203.3M), forniceal_palpebral
!python Segmentation/scripts/train_pretrained/train_cnn_strong_convnext_large_unet_forniceal_palpebral.py
sync_outputs()

Using device: cuda
Model: cnn_strong_convnext_large_unet_forniceal_palpebral (pretrained build_model)
Dataset: AlignedConjunctivaSegmentationDataset (tissue_type=forniceal_palpebral)
Image size: 256
[I 2026-08-10 18:53:37,568] A new study created in memory with name: no-name-34e6ad79-3cd1-4d7c-9da6-acc3c61afadf
[cnn_strong_convnext_large_unet_forniceal_palpebral | Trial 0] New best overall val_dice=0.2149 -> saved /kaggle/working/eyes-defy-anemia/Segmentation/outputs/checkpoints/best_cnn_strong_convnext_large_unet_forniceal_palpebral.pth (fp16)
[cnn_strong_convnext_large_unet_forniceal_palpebral | Trial 0 | loss_fn=bce_dice] Epoch  1/30 - train_loss=0.8982 val_loss=0.9500 val_dice=0.2149 val_iou=0.1217 val_precision=0.1217 val_recall=0.9994
[cnn_strong_convnext_large_unet_forniceal_palpebral | Trial 0] New best overall val_dice=0.3046 -> saved /kaggle/working/eyes-defy-anemia/Segmentation/outputs/checkpoints/best_cnn_strong_convnext_large_unet_forniceal_palpebral.pth (fp16)
[cnn_strong

In [13]:
# Strong tier, combo 17/18 -- Swin-Large + UperNet (Transformer, ~233.9M), palpebral
!python Segmentation/scripts/train_pretrained/train_transformer_strong_swin_large_upernet_palpebral.py
sync_outputs()

Using device: cuda
Model: transformer_strong_swin_large_upernet_palpebral (pretrained build_model)
Dataset: AlignedConjunctivaSegmentationDataset (tissue_type=palpebral)
Image size: 512
[I 2026-08-10 20:46:47,106] A new study created in memory with name: no-name-2ee7ca23-8e26-4f4d-a972-a7d2f0f5b100
config.json: 8.98kB [00:00, 18.8MB/s]
model.safetensors: 100%|██████████████████████| 940M/940M [00:06<00:00, 146MB/s]
Loading weights: 100%|█| 535/535 [00:00<00:00, 1226.51it/s, Materializing param=
UperNetForSemanticSegmentation LOAD REPORT from: openmmlab/upernet-swin-large
Key                              | Status   |                                                                                                   
---------------------------------+----------+---------------------------------------------------------------------------------------------------
auxiliary_head.classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([150]) vs model:torch.Size([1])          

In [14]:
# Strong tier, combo 18/18 -- Swin-Large + UperNet (Transformer, ~233.9M), forniceal_palpebral
!python Segmentation/scripts/train_pretrained/train_transformer_strong_swin_large_upernet_forniceal_palpebral.py
sync_outputs()

Using device: cuda
Model: transformer_strong_swin_large_upernet_forniceal_palpebral (pretrained build_model)
Dataset: AlignedConjunctivaSegmentationDataset (tissue_type=forniceal_palpebral)
Image size: 512
[I 2026-08-10 20:48:13,419] A new study created in memory with name: no-name-14c4c428-8b7b-4e2a-b30b-6c45de28d794
Loading weights: 100%|█| 535/535 [00:00<00:00, 1410.89it/s, Materializing param=
UperNetForSemanticSegmentation LOAD REPORT from: openmmlab/upernet-swin-large
Key                              | Status   |                                                                                                   
---------------------------------+----------+---------------------------------------------------------------------------------------------------
decode_head.classifier.bias      | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([150]) vs model:torch.Size([1])                      
decode_head.classifier.weight    | MISMATCH | Reinit due to size mismatch ckpt: torch.

## Done — what to download

Everything from this tier is consolidated at `/kaggle/working/outputs/` (checkpoints, logs) and zipped to `/kaggle/working/segmentation_sweep_results.zip`. Both are visible in this notebook version's **Output** tab once you Save Version -> Save & Run All — download the zip directly from there. This is the last of the 3 tiers — once you have all three zips downloaded (Base, Mid, Strong), you have all 18 combos' checkpoints/logs/plots.

A failed `!python ...` cell does **not** halt "Run All" — check each script's own printed output (or the saved `Segmentation/outputs/logs/*_study_summary.json` files) after this finishes, not just whether the notebook run itself completed.

In [15]:
from pathlib import Path

print('Final contents of /kaggle/working/outputs:')
for f in sorted(Path('/kaggle/working/outputs').rglob('*')):
    if f.is_file():
        print(f'  {f.relative_to("/kaggle/working/outputs")}  ({f.stat().st_size / 1e6:.2f} MB)')

zip_path = Path('/kaggle/working/segmentation_sweep_results.zip')
print(f"\nZip archive: {zip_path}  ({zip_path.stat().st_size / 1e6:.2f} MB)")

Final contents of /kaggle/working/outputs:
  checkpoints/best_cnn_strong_convnext_large_unet_forniceal_palpebral.pth  (406.69 MB)
  checkpoints/best_cnn_strong_convnext_large_unet_palpebral.pth  (406.69 MB)
  checkpoints/best_hybrid_strong_transunet_forniceal_palpebral.pth  (242.06 MB)
  checkpoints/best_hybrid_strong_transunet_palpebral.pth  (242.05 MB)
  logs/cnn_strong_convnext_large_unet_forniceal_palpebral_study_summary.json  (0.00 MB)
  logs/cnn_strong_convnext_large_unet_forniceal_palpebral_test_per_patient.csv  (0.00 MB)
  logs/cnn_strong_convnext_large_unet_forniceal_palpebral_trials.csv  (0.07 MB)
  logs/cnn_strong_convnext_large_unet_palpebral_study_summary.json  (0.00 MB)
  logs/cnn_strong_convnext_large_unet_palpebral_test_per_patient.csv  (0.00 MB)
  logs/cnn_strong_convnext_large_unet_palpebral_trials.csv  (0.08 MB)
  logs/hybrid_strong_transunet_forniceal_palpebral_study_summary.json  (0.00 MB)
  logs/hybrid_strong_transunet_forniceal_palpebral_test_per_patient.csv  (0.